# Track 01 — EEG-to-Image · pipeline proof

**What this notebook proves:** that we can pull the data, pull a model, train it, evaluate it
with the competition's metric, and package a submission — end to end, on Colab, without surprises.

**What it does not do:** chase performance. Every model here is a stock baseline.

---

### The task

Given one EEG epoch recorded while a participant viewed a natural image, predict a **1536-d
embedding** in frozen `facebook/dinov2-giant` space, then rank held-out candidate images by
similarity. Training and test images do not overlap — the shift is *cross-stimulus*.

| | |
|---|---|
| Ranking metric | Top-5 retrieval accuracy against the full held-out gallery |
| NeuralBench key | `test/full_retrieval/top5_acc_subject-agg` |
| Target space | `facebook/dinov2-giant`, relative depth 0.6667, mean token pooling, imsize 518 |
| Hidden cohort | 11 participants, 32 ch @ 256 Hz (Alljoined) |

⚠️ `val/batch_top5_acc` ranks **within a batch** only. It reads far higher and is not comparable
to the competition score. Never quote it.

On the hardware: the challenge site describes the cohort as 32-channel Emotiv. The Alljoined-1.6M
paper (arXiv:2508.18571) describes a 32-channel consumer-grade **wet** electrode system at ~$2.2k,
and the recordings carry standard 10-10 electrode names rather than vendor labels. Same count and
rate either way; the naming matters because it means a standard montage resolves cleanly.

## Two things to know before running

**1. Subsetting applies to Stage 0 only.** `FRACTION` and the byte budget govern how many *records*
EEGDash pulls. Stage 1 uses NeuralBench, whose downloader works at corpus granularity with no subset
option, so it fetches all of Alljoined-1.6M regardless. Inside Stage 1 the only data reduction is
`-d/--debug`, which NeuralBench applies itself.

**2. Track 1 subsetting has a trap.** The score is retrieval against the held-out gallery, so chance
is `k / gallery_size`. Shrinking the gallery inflates the score for reasons unrelated to the model:

| gallery | chance Top-5 |
|---|---|
| 16,740 | 0.03% |
| 200 | 2.50% |
| 10 | **50.00%** |

So Stage 0 subsets the training side and leaves the test gallery whole, and `assert_gallery_intact`
fails the run if a subset reaches the test split. Every Top-5 number is reported with its gallery
size attached, because one without the other means nothing.

> Cells marked `# VERIFY` were written from docs and not yet confirmed against a run. Several have
> since been resolved; the remainder are noted where they appear.

---

# Setup — run this first, everything below depends on it

Built for **Run All**. The two flags below decide how far it goes; nothing else needs editing.

| flag | off (default) | on |
|---|---|---|
| `RUN_HEAVY` | Stage 0 only: ~60 MB, CPU, minutes | adds the 7.7 GB download, prepare, and debug run |
| `RUN_FULL_TRAIN` | no full training | adds the full EEGNet baseline, hours |

With both off, Run All completes the whole discovery path and skips every expensive cell with a
printed reason. Nothing errors, nothing blocks on input.

Drive mounts **before** the installs, deliberately: installing NeuralBench downgrades packages Colab
pins, including `requests`, and `google.colab` imports can fail afterwards.

In [ ]:
#@title Run configuration { display-mode: "form" }
#@markdown Pick a mode, then **Runtime > Run all**. Nothing else needs editing.
#@markdown 
#@markdown - **discovery** - CPU, ~60 MB, minutes. Proves data access, montage and events.
#@markdown - **download** - GPU, ~13 GB, hours. Adds the corpus, the prepare cache and `--debug`.
#@markdown - **train** - GPU, many hours. Adds the full EEGNet baseline.

MODE = "0 - discovery (CPU, ~60 MB, minutes)" #@param ["0 - discovery (CPU, ~60 MB, minutes)", "1 - download + prepare + debug (GPU, ~13 GB, hours)", "2 - full training (GPU, many hours)"]
USE_DRIVE = True #@param {type:"boolean"}
BUDGET_MB = 60 #@param {type:"slider", min:10, max:500, step:10}
FRACTION = 0.02 #@param {type:"number"}
SEED = 33 #@param {type:"integer"}

#@markdown ---
#@markdown `BUDGET_MB` and `FRACTION` govern Stage 0 record selection only. NeuralBench
#@markdown downloads at corpus granularity and ignores both.

_level = int(MODE.split(' - ')[0])
RUN_HEAVY = _level >= 1
RUN_FULL_TRAIN = _level >= 2

print(f'mode           : {MODE}')
print(f'RUN_HEAVY      : {RUN_HEAVY}')
print(f'RUN_FULL_TRAIN : {RUN_FULL_TRAIN}')
print(f'stage 0 budget : {BUDGET_MB} MB, fraction {FRACTION}, seed {SEED}')
if not RUN_HEAVY:
    print('\nHeavy cells will skip with a printed reason. Nothing will error.')

In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/neurips26-eeg')
else:
    ROOT = Path('/content/neurips26-eeg')

DATA_DIR = ROOT / 'data'
SAVE_DIR = ROOT / 'results'
CACHE_LOCAL = Path('/content/nb_cache')     # rebuildable, so keep it off slow Drive
EEGDASH_CACHE = Path('/content/eegdash_cache')
for d in (DATA_DIR, SAVE_DIR, CACHE_LOCAL, EEGDASH_CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(ROOT / 'hf')
os.environ['EEGDASH_CACHE_DIR'] = str(EEGDASH_CACHE)
CACHE = EEGDASH_CACHE
print('ROOT      :', ROOT)
print('free (data):', round(shutil.disk_usage(DATA_DIR).free / 1e9, 1), 'GB')
print('free (local):', round(shutil.disk_usage('/content').free / 1e9, 1), 'GB')

In [ ]:
REPO = 'https://github.com/AGRamirezz/Neurips26-eeg-foundation-model.git'
SRC = Path('/content/repo/src')

if Path('../src/miniload.py').exists():
    SRC = Path('../src').resolve()
elif not SRC.exists():
    subprocess.run(['git', 'clone', '-q', REPO, '/content/repo'], check=True)
sys.path.insert(0, str(SRC))
print('helpers:', sorted(f.name for f in SRC.glob('*.py')))

In [ ]:
%pip install -q 'eegdash>=0.9.1' neuralbench==0.3.1 'transformers' 'torchvision' 'pillow'

In [ ]:
# Fail loudly here rather than 40 cells later.
import importlib
for mod in ('eegdash', 'neuralbench', 'mne'):
    try:
        importlib.import_module(mod)
        print(f'{mod:12} ok')
    except ImportError as e:
        raise SystemExit(f'{mod} did not install: {e}. Restart the runtime and re-run Setup.')

print('python  :', sys.version.split()[0])
assert sys.version_info >= (3, 12), 'NeuralBench needs Python >= 3.12'

### NeuralBench storage

Environment variables are **not** read. With no terminal to prompt from, NeuralBench silently
defaults `DATA_DIR`, `CACHE_DIR` and `SAVE_DIR` to `/tmp/neuralbench`, which Colab discards on
restart. The mechanism is a config file at `~/.neuralbench/config.json`.

`DATA_DIR` goes to Drive because a 7.7 GB download is expensive to refetch. `CACHE_DIR` stays on
local disk because Drive is slow for many small files and the cache rebuilds in minutes.

In [ ]:
CFG_PATH = Path.home() / '.neuralbench' / 'config.json'
CFG_PATH.parent.mkdir(parents=True, exist_ok=True)
CFG_PATH.write_text(json.dumps({
    'USER': os.environ.get('USER', 'root'),
    'ENTITY_NAME': os.environ.get('USER', 'root'),
    'PROJECT_NAME': 'neuralbench',
    'DATA_DIR': str(DATA_DIR),
    'CACHE_DIR': str(CACHE_LOCAL),
    'SAVE_DIR': str(SAVE_DIR),
    'WANDB_HOST': '', 'SLURM_PARTITION': '', 'SLURM_CONSTRAINT': '',
    'N_CPUS': os.cpu_count() or 2, 'CLUSTER': 'auto',
}, indent=2))
os.environ['NEURALBENCH_CONFIG'] = str(CFG_PATH)

# The CLI runs as a subprocess, so check what a fresh interpreter resolves.
out = subprocess.run([sys.executable, '-c',
    'from neuralbench import config_manager as c; import json; print(json.dumps(c.get_config()))'],
    capture_output=True, text=True)
resolved = json.loads(out.stdout.strip().splitlines()[-1])
for k in ('DATA_DIR', 'CACHE_DIR', 'SAVE_DIR'):
    print(f'{k:10} {resolved[k]}')
assert '/tmp/' not in resolved['DATA_DIR'], 'config not picked up: DATA_DIR is still ephemeral'
print('\nconfig ok')

---

# Stage 0 — smoke test on a few recordings

**Run this first.** CPU only, no GPU, no Drive. It works on a laptop.

Alljoined-1.6M is on EEGDash as **`nm000134`**: 20 subjects, 1525 recordings, 32 ch @ 256 Hz. One
317 s recording is about 6 MB, so a 60 MB budget buys roughly six of them, spread across six
subjects. Same corpus and hardware as the hidden evaluation cohort.

(EEGDash reports the corpus at 8.8 GB and 129 h; the challenge dataset table says 7.7 GB and 130 h.
Same data, different accounting. Neither figure matters at this scale.)

Record selection is the only way to touch a large corpus cheaply, because NeuralBench's own
downloader has no subset option.

Stage 0's job is **discovery**: print what the data actually looks like, so the stubbed cells in
Stage 1 can be written against reality rather than guessed from docs.

⚠️ Alljoined-1.6M is **CC-BY-NC-ND-4.0**, non-commercial *and* no-derivatives. Stricter than the
other Track 1 corpora. Worth checking before anything derived from it is published.

In [ ]:
# Moved to the Setup section at the top. Nothing to run here.

In [ ]:
# Moved to the Setup section at the top. Nothing to run here.

### 0.1 — What is in the dataset

Query first, download nothing. This tells us the subject/session/task vocabulary to select on.

In [ ]:
from eegdash import EEGDash

DATASET = 'nm000134'   # Alljoined-1.6M

records = EEGDash().find({'dataset': DATASET})
print('records:', len(records))

# Keep the converted BIDS files, not the original source format.
records = [r for r in records if r['bids_relpath'].startswith('sub-')]
print('bids records:', len(records))
print('\nfields on a record:')
print(json.dumps({k: str(v)[:60] for k, v in records[0].items()}, indent=2))

In [ ]:
# Vocabulary we can select on.
from collections import Counter
for field in ('subject', 'session', 'task', 'run'):
    vals = Counter(r.get(field) for r in records)
    print(f'{field:>8}: {len(vals)} unique -> {sorted(str(v) for v in vals)[:8]}')

### 0.2 — Pull a few recordings, under a byte budget

`select_under_budget` reads `ntimes` and `nchans` from the record metadata, so the size of a
selection is known before anything transfers.

`spread_by='subject'` matters more than it looks. Record lists arrive grouped by subject, so taking
the first six in order gives six recordings from one person: enough to prove the code runs, useless
for seeing whether it handles variation. Interleaving costs identical bytes.

In [ ]:
from miniload import select_under_budget, estimate_mb, host_limits, epochs_mb, targets_mb

print('host:', {k: (v if isinstance(v, str) else round(v, 1)) for k, v in host_limits().items()})

# Budget first, download second. Spread across subjects so the batch exercises
# variation, not just one person's recordings.
sel = select_under_budget(records, budget_mb=BUDGET_MB, spread_by='subject')
picked = sel.records

print(f'\n{len(picked)} records, ~{sel.est_mb:.0f} MB est '
      f'({len({r["subject"] for r in picked})} subjects, {sel.skipped_over_budget} skipped)')
for r in picked:
    print(f"  {estimate_mb(r):5.1f} MB  {r['bids_relpath']}")

**Memory, measured rather than guessed.** One 317 s recording at 32 ch / 256 Hz is 10.4 MB as
float32. Epoched into 1 s windows it becomes ~42 MB per recording, and the full set of 16,740
DINOv2-giant targets is 103 MB (which matches the ~100 MB the NeuralBench docs quote for
THINGS-EEG2, a useful cross-check).

None of that is near a Colab limit. The whole corpus held in RAM would be ~15.6 GB, which is why we
do not hold it. The one item of consequence is the DINOv2-giant checkpoint at ~4.5 GB; on a GPU
runtime it lands in VRAM, on a CPU runtime it competes with everything else.

In [ ]:
from eegdash import EEGDashDataset

ds = EEGDashDataset(records=picked, cache_dir=CACHE)
print(ds.description.to_string(index=False))

### 0.3 — Inspect one recording

Signal shape, sampling rate, channel names, and the event schema.

**Outcome:** 32 channels at 256 Hz, 317 s, and `montage_present: False`. The events turned out not
to carry per-image identity at all, which 0.5 and 0.6 follow up.

In [ ]:
raw = ds.datasets[0].raw          # VERIFY: attribute name for the underlying MNE Raw
print('sfreq   :', raw.info['sfreq'])
print('n_chans :', len(raw.ch_names))
print('channels:', raw.ch_names)
print('duration:', raw.n_times / raw.info['sfreq'], 's')
print('has montage positions:', raw.get_montage() is not None)

In [ ]:
import pandas as pd

# The events sidecar is what maps an EEG epoch to the image that was shown.
ev = raw.annotations.to_data_frame() if len(raw.annotations) else None
print('annotations:', len(raw.annotations))
if ev is not None:
    print(ev.head(10).to_string())
    print('\ncolumns:', list(ev.columns))

# VERIFY: locate the stimulus-identity column and the stimuli.tsv mapping for Alljoined.
# THINGS-EEG2 used tot_img_number + stimuli.tsv; Alljoined's schema is unknown here.

### 0.4 — Consolidated report

One cell summarising the run. This is what resolved the montage gap, the `02old` session, and the
event schema.

In [ ]:
import platform, sys

report = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'n_records_total': len(records),
    'select_fields': {f: sorted({str(r.get(f)) for r in records})[:10]
                      for f in ('subject', 'session', 'task', 'run')},
    'record_keys': list(records[0].keys()),
    'sfreq': raw.info['sfreq'],
    'ch_names': raw.ch_names,
    'n_annotations': len(raw.annotations),
    'event_columns': list(ev.columns) if ev is not None else None,
    'montage_present': raw.get_montage() is not None,
}
print(json.dumps(report, indent=2, default=str))

---

## 0.5 — Fixes from the first run

Three findings from the Stage 0 report, in order of importance.

**Montage is absent.** `montage_present: False`. REVE encodes 3-D electrode coordinates, so with no
montage every channel becomes `INVALID_VALUE` and the model's main advantage silently disappears.
The 32 names are standard 10-10 and all resolve, verified offline at 32/32 with no NaNs. Use
**`standard_1020`**: `standard_1005` resolves equally well but is deprecated from MNE 1.14. Same
pattern the stock Track 3 config uses for Sleep-EDF.

**Session `02old` exists** alongside `01`-`04`. A superseded duplicate. Exclude it, along with
records flagged `_has_missing_files`.

**Events are not in `raw.annotations`.** 61 annotations over 317 s, a median gap of 5.6 s, far too
sparse for a corpus averaging ~1050 image trials per recording.

In [ ]:
# Record filtering, and a size budget computed before anything downloads.
EXCLUDE_SESSIONS = {'02old'}

clean = [
    r for r in records
    if not r.get('_has_missing_files')
    and r.get('session') not in EXCLUDE_SESSIONS
]
print(f'{len(records)} records -> {len(clean)} after filtering')

def est_mb(r):
    # float32 per sample per channel; a rough but useful pre-download budget.
    n = r.get('ntimes') or 0
    c = r.get('nchans') or 0
    return n * c * 4 / 1e6

tot = sum(est_mb(r) for r in clean)
print(f'full filtered set: ~{tot/1000:.1f} GB across {len(clean)} records')
print(f'median record: ~{sorted(est_mb(r) for r in clean)[len(clean)//2]:.1f} MB')

In [ ]:
# Set the montage explicitly and confirm every channel resolves to real coordinates.
import numpy as np, mne

MONTAGE = 'standard_1020'   # 32/32 resolve; standard_1005 is deprecated from MNE 1.14

raw.set_montage(mne.channels.make_standard_montage(MONTAGE), match_case=False)

pos = raw.get_montage().get_positions()['ch_pos']
P = np.array([pos[c] for c in raw.ch_names])
assert not np.isnan(P).any(), 'some channels still have no coordinates'
print(f'{len(P)}/{len(raw.ch_names)} channels positioned, no NaNs')
print(f'posterior (y<0): {(P[:,1]<0).sum()}  anterior (y>0): {(P[:,1]>0).sum()}')
# Heavily posterior - a visual-task montage, unlike the frontal/temporal-heavy
# clinical layouts most EEG foundation models pretrain on.

### Annotations decoded

`description` holds **four** comma-separated fields: `<condition>,<stim_id>,<block>,<trial>`. The
TSV writes it as `behav,3,-1,21`, which reads as a three-field record with a decimal comma until you
count. Conditions are `behav` (ids 2, 3) and `oddball` (id 16740); block is `-1` throughout.

Read that way the trial index is monotonic: 21, 42, 63, 84, then 381, 402, 423, mostly stepping by
21. A behavioural probe lands every 21 image trials, so 61 markers spans roughly 1280 trials. The
image presentations happened; this file does not enumerate them.

`16740` is the THINGS catalogue size, so the oddball id is a sentinel rather than a real image
index. The corpus is part of the THINGS initiative (arXiv:2508.18571), recorded on a 32-channel
consumer-grade **wet** system, which is why the channel names are standard 10-10 rather than a
vendor's own labels.

⚠️ Use `raw.annotations.onset` for seconds. `to_data_frame()` returns absolute datetimes here
because `meas_date` is set, which is useless for epoching.

In [ ]:
import pandas as pd

ann = pd.DataFrame({
    'onset_s': raw.annotations.onset,          # seconds, not datetimes
    'duration': raw.annotations.duration,
    'description': raw.annotations.description,
})
# Four fields: condition, stimulus id, block, trial index. The TSV writes the last
# two with a comma between them, which reads as a decimal comma until you count.
parts = ann['description'].str.split(',', expand=True)
ann[['condition', 'stim_id', 'block', 'trial']] = parts.iloc[:, :4]
for c in ('stim_id', 'block', 'trial'):
    ann[c] = pd.to_numeric(ann[c], errors='coerce')

print(ann.head(12).to_string())
print('\nconditions:', ann['condition'].value_counts().to_dict())
print('blocks:', ann['block'].unique())
print('trial index: %s -> %s, monotonic=%s'
      % (ann['trial'].min(), ann['trial'].max(), ann['trial'].is_monotonic_increasing))

### Find the stimulus identity

**Outcome: not present.** The `events.tsv` for `sub-01/ses-02/run-13` has the same 61 rows as the
annotations, columns `onset, duration, trial_type, value, sample`, and no per-image entries. Cells
below are kept because they are the right probe to re-run against other recordings.

In [ ]:
import pandas as pd

ev_files = sorted(CACHE.rglob('*_events.tsv'))
print(f'{len(ev_files)} events.tsv found')
for f in ev_files[:3]:
    print('  ', f.relative_to(CACHE))

assert ev_files, 'no events.tsv in cache - check what EEGDash actually downloaded'
ev = pd.read_csv(ev_files[0], sep='\t')
print(f'\nshape: {ev.shape}')
print('columns:', list(ev.columns))
print(ev.head(10).to_string())

In [ ]:
# Which column identifies the image? Look for one with many distinct values.
for c in ev.columns:
    n = ev[c].nunique()
    print(f'{c:>24}  {n:>6} unique  e.g. {list(ev[c].dropna().unique()[:4])}')

# Also: what are the 61 annotations, if not stimuli?
print('\nannotation descriptions:', raw.annotations.description[:20].tolist())

In [ ]:
# Any stimulus->file mapping at the dataset root (THINGS-EEG2 used stimuli.tsv).
for pat in ('stimuli.tsv', 'participants.tsv', '*_events.json', 'dataset_description.json'):
    for f in sorted(CACHE.rglob(pat))[:2]:
        print('--', f.relative_to(CACHE))
        if f.suffix == '.tsv':
            print(pd.read_csv(f, sep='\t').head(5).to_string())
        else:
            print(json.dumps(json.load(open(f)), indent=2)[:600])

---

## 0.6 — Where is the image stream? (parked)

**Result: durations are unimodal.** 255-409 s, median 308, one peak. There is no separate class of
image runs, so per-image identity is not a matter of picking different recordings.

**The per-file event count test did not actually run.** `EEGDashDataset` is lazy: it fetches an EDF
when you touch `.raw`. Section 0.6 built the dataset but never accessed it, so the glob found only
the single file Stage 0 had already pulled. Cell below forces the download if we return to this.

**Parked, because Stage 1 does not depend on it.** NeuralBench ships a registered `xu2025alljoined`
config for the `eeg image` task and handles target extraction itself. Hand-rolling the stimulus
mapping is only needed if that config turns out to be broken or absent. Run Stage 1 first.

Open question recorded in `PLAN.md` §4b.

In [ ]:
# Forces the fetch that 0.6 assumed. Only needed if we return to hand-rolled extraction.
# for d in ds_probe.datasets:
#     _ = d.raw
# rows = [(f.name, len(pd.read_csv(f, sep='\t'))) for f in sorted(CACHE.rglob('*_events.tsv'))]
# print(pd.DataFrame(rows, columns=['file', 'n_events']).to_string())

In [ ]:
# Duration distribution across all records. Costs nothing: it is in the metadata already.
import numpy as np

durs = np.array([r.get('duration_seconds') or 0 for r in clean], dtype=float)
print(f'{len(durs)} records')
print('percentiles (s):', {q: round(float(np.percentile(durs, q)), 1)
                           for q in (0, 10, 50, 90, 100)})

hist, edges = np.histogram(durs, bins=12)
for h, lo, hi in zip(hist, edges[:-1], edges[1:]):
    print(f'{lo:7.0f}-{hi:7.0f}s  {h:5d}  {"#" * min(60, h // 5)}')

In [ ]:
# If durations are bimodal, the long mode is where the image trials live.
long_cut = float(np.percentile(durs, 75))
long_records = [r for r in clean if (r.get('duration_seconds') or 0) > long_cut]
print(f'{len(long_records)} records longer than {long_cut:.0f}s')

from collections import Counter
print('by session:', Counter(r['session'] for r in long_records))
print('by run    :', dict(sorted(Counter(r['run'] for r in long_records).items())[:12]))

### Sample across runs and count events in each

**This did not actually run.** `EEGDashDataset` is lazy and fetches an EDF only when `.raw` is
touched, so the glob found only the file Stage 0 had already pulled. The commented cell in 0.6
forces the fetch if we return to this.

In [ ]:
import pandas as pd

# subj came from an earlier hand-picked selection that budget-based selection replaced.
subj = picked[0]['subject']

probe = []
seen_runs = set()
for r in clean:
    if r['subject'] != subj or r['run'] in seen_runs:
        continue
    seen_runs.add(r['run'])
    probe.append(r)
    if len(probe) == 6:
        break

for r in probe:
    print(f"ses-{r['session']} run-{r['run']}  {r.get('duration_seconds'):>7.1f}s  {r['bids_relpath']}")

In [ ]:
# Parked (see 0.6): re-downloads without resolving anything. Kept for reference.
# ds_probe = EEGDashDataset(records=probe, cache_dir=CACHE)
# 
# rows = []
# for f in sorted(CACHE.rglob('*_events.tsv')):
#     ev = pd.read_csv(f, sep='\t')
#     tt = ev['trial_type'].astype(str) if 'trial_type' in ev else pd.Series(dtype=str)
#     rows.append({
#         'file': f.name,
#         'n_events': len(ev),
#         'conditions': tt.str.split(',').str[0].value_counts().to_dict(),
#         'columns': list(ev.columns),
#     })
# print(pd.DataFrame(rows).to_string())

In [ ]:
# Parked (see 0.6): re-downloads without resolving anything. Kept for reference.
# # If some file has ~1000 rows, inspect it. That is the stimulus stream.
# big = max(sorted(CACHE.rglob('*_events.tsv')), key=lambda f: len(pd.read_csv(f, sep='\t')))
# ev = pd.read_csv(big, sep='\t')
# print(big.name, ev.shape)
# print(ev.head(15).to_string())
# for c in ev.columns:
#     print(f'{c:>12}  {ev[c].nunique():>6} unique')

---

# Stage 1 — NeuralBench, GPU, competition harness

**This is not a couple of data points.** NeuralBench's downloader works at corpus granularity, so
`--download` fetches all of Alljoined-1.6M. The Stage 0 budget does not apply. Expect 7.7 GB of EEG,
4.5 GB for DINOv2-giant, plus the prepare cache.

**Two stops, not one run.**

1. **Through the config-confirmation cell, then stop.** Storage is the live risk: env vars are not
   read, and an unconfigured run puts everything in `/tmp/neuralbench`, which Colab discards on
   restart. Confirm `DATA_DIR` resolves to Drive before starting a 7.7 GB transfer.
2. **Download, prepare, then `--debug -m eegnet`.** Prepare is the long part: it runs DINOv2 over
   every unique stimulus and is the only `--prepare` of the four tracks needing a GPU. The docs
   quote its timings across 10-128 SLURM jobs; on Colab it runs serially, so expect materially
   longer. `--debug` afterwards is the actual end-to-end proof and takes minutes.

EEGNet before REVE, because EEGNet needs no gated weights and gets you a green pipeline while the
HuggingFace approval for `brain-bzh/reve-base` is pending.

**Still inert:** the `check_model` call, the subset and gallery guards, and the inference timer. The
chance control is now real (`-m chance`). A score from this pass has few gates behind it, so treat
it as evidence the plumbing works and nothing more.

---
## [0] Environment check

Fail loudly and early. NeuralBench needs Python >= 3.12 and has **no CPU fallback**: every training
run, `--debug` included, requires a working GPU. Colab currently ships Python 3.13.15, so the
version requirement is satisfied.

In [ ]:
import sys, subprocess

print('Python:', sys.version)
assert sys.version_info >= (3, 12), (
    f'NeuralBench requires Python >=3.12, got {sys.version_info.major}.{sys.version_info.minor}. '
    'Fallback: uv venv --python 3.12'
)

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU. NeuralBench has no CPU fallback.'
print('torch:', torch.__version__, '| cuda:', torch.version.cuda)
# A driver/torch mismatch raises here rather than failing silently later.
print('capability:', torch.cuda.get_device_capability(0))
print('device:', torch.cuda.get_device_name(0))

---
## [1] Persistent paths

Colab runtimes are ephemeral and NeuralBench defaults everything to `/tmp/neuralbench`, so paths
have to be set deliberately. The mechanism is a config file, not environment variables: see `[2]`.

Mount Drive **before** installing. The install downgrades packages Colab pins, including `requests`,
which `google-colab` depends on, so `google.colab` imports can fail after a restart.

Track 1 needs room for Alljoined-1.6M (~7.7 GB), the DINOv2-giant checkpoint (~4.5 GB), the
preprocessing cache, and one frozen embedding per unique stimulus (~103 MB for the full THINGS set).

In [ ]:
import os
from pathlib import Path

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/neurips26-eeg')
else:
    ROOT = Path('/content/neurips26-eeg')

DATA_DIR  = ROOT / 'data'
CACHE_DIR = ROOT / 'cache'
SAVE_DIR  = ROOT / 'results'
for d in (DATA_DIR, CACHE_DIR, SAVE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# HF checkpoints are large; keep them on Drive too so they survive a restart.
os.environ['HF_HOME'] = str(ROOT / 'hf')

import shutil
free_gb = shutil.disk_usage('/content').free / 1e9
print(f'ROOT={ROOT}\nfree on /content: {free_gb:.1f} GB')

---
## [2] Install

Pin versions so a green run stays green.

The install reports dependency conflicts against packages Colab pins (`requests`, `decorator`,
`opentelemetry`). Harmless within a session that mounted Drive first. The braindecode
`standard_1020` deprecation warnings come from braindecode's own code, not ours.

In [ ]:
# Moved to the Setup section at the top. Nothing to run here.

In [ ]:
# Moved to the Setup section at the top. Nothing to run here.

**Verified CLI surface** (from the help output above):

```
neuralbench [opts] {eeg,emg,fmri,meg} {task}
  --dataset D   loads datasets/{D}.yaml over the base config
  -m MODEL      chance dummy eegnet atcnet deep4net eegconformer shallow_fbcsp_net ctnet
                bendr biot cbramod labram luna reve mae  |  groups: all_classic all_fm all_baseline
  -w PRESET     linear_probe_flatten linear_probe_mean attentive_probe
                lora_r4_flatten lora_r32_flatten finetune_mean finetune_flatten
                (foundation models only)
  -d --debug    smaller config, runs locally, infra.mode='force'
  -p --prepare / --download / --plot-cached / -g --grid / -f --force / -r --retry
```

Three corrections to what this notebook assumed:

- **`chance` is a model.** The chance control needs no hand-wiring; `-m chance` puts it through the
  real scoring path. That turns the G1 anchor from a stub into a genuine check.
- **LoRA presets are `lora_r4_flatten` and `lora_r32_flatten`**, not `-w lora`. Two ranks to compare.
- **LUNA's CLI name is `luna`**, not `NtLuna`.

`image` is registered for `Gifford2022Large, Grootswagers2022Human, Xu2024Alljoined, Xu2025Alljoined`,
confirming `--dataset xu2025alljoined`.

Still unverified: nothing in the CLI sets DATA_DIR / CACHE_DIR / SAVE_DIR, so the env vars above are
still a guess. If the download lands somewhere unexpected, `config_manager.setup_config` is the
documented alternative.

⚠️ The install downgraded packages Colab pins (`requests`, `decorator`, `opentelemetry`). Harmless
in this session because Drive was already mounted, but after a runtime restart `google.colab`
imports may break. Mount Drive before installing, as this notebook does.

### Confirm the config took

Seconds to check, and worth doing before committing to 7.7 GB. The CLI runs as a subprocess, so
what matters is what a fresh interpreter resolves, not what this kernel already imported.

`DATA_DIR` should point at Drive. If it still reads `/tmp/neuralbench`, the config file was not
picked up and the download would be lost on the next runtime restart.

In [ ]:
# Moved to the Setup section at the top. Nothing to run here.

---
## [2b] Subsetting config

Imported here so the helpers are available, but **these govern Stage 0 only**. NeuralBench does its
own data handling and does not consult them; inside Stage 1 the only reduction lever is `--debug`.

Two independent levers, because download and load are different problems:

- **record selection** happens *before* download, the only way to touch a large corpus from a Colab
  disk. EEGDash queries exact recordings.
- **group subsetting** happens *after* download, a deterministic fraction of what is on disk,
  sampled over whole groups so split semantics survive.

In [ ]:
from subset import (
    SubsetSpec, subset_groups, chance_top_k, describe,
    assert_disjoint, assert_gallery_intact, TRACK1_RULE,
)

FRACTION = 0.02          # 1.0 for a full run
SEED     = 33

spec = SubsetSpec(
    fraction=FRACTION,
    seed=SEED,
    group_key='subject',   # NOT image_id - see the gallery trap above
    notes=TRACK1_RULE,
)
print(spec)

---
## [3] Data — Alljoined-1.6M

Registered as `xu2025alljoined`, confirmed present in the task's dataset list alongside
`Gifford2022Large`, `Grootswagers2022Human` and `Xu2024Alljoined`. 20 participants, 32 ch @ 256 Hz.

The size guard below exists because the *default* dataset for this task is THINGS-EEG2 at
**220 GB**. Omitting `--dataset` starts that download. Do not omit it.

In [ ]:
import shutil

DATASET = 'xu2025alljoined'   # NOT the default. Default = Gifford2022Large @ 220 GB.

for label, path, need in (('DATA_DIR (Drive)', DATA_DIR, 7.7 + 4.5),
                          ('CACHE_DIR (local)', CACHE_LOCAL, 6.0)):
    free = shutil.disk_usage(path).free / 1e9
    status = 'ok' if free > need else 'TOO SMALL'
    print(f'{label:20} {str(path):34} free {free:6.1f} GB  need ~{need:4.1f} GB  {status}')
    assert free > need, f'{label} has {free:.1f} GB free, needs ~{need:.1f} GB'

print('\nDownload is resumable: it skips files already on disk.')

In [ ]:
if RUN_HEAVY:
    !neuralbench eeg image --dataset {DATASET} --download
else:
    print('skipped: set RUN_HEAVY = True in Setup to fetch ~7.7 GB')

### [3b] Prepare the cache

**This is the only `--prepare` of the four tracks that needs a GPU** — it runs DINOv2-giant over
every unique stimulus to build the frozen target embeddings, alongside the usual window
preprocessing. Embeddings are content-keyed and shared across image tasks, so this cost is paid
once.

The docs quote SLURM-parallel timings (10 and 128 jobs). On Colab this runs serially and in-process,
so expect it to take proportionally longer.

In [ ]:
import time

if RUN_HEAVY:
    t0 = time.time()
    !neuralbench eeg image --dataset {DATASET} --prepare
    print(f'prepare took {(time.time() - t0) / 60:.1f} min')
else:
    print('skipped: needs RUN_HEAVY and a completed download')

---
## [4] Montage and input adaptation

Alljoined is a 32-channel consumer-grade **wet** system at 256 Hz, with standard 10-10 electrode
names rather than vendor labels. Encoder native rates differ: REVE is 200 Hz, LUNA is 256 Hz, so log
the resample path explicitly rather than letting a silent mismatch cost accuracy.

Confirm channel positions resolve. The recordings ship with **no montage**, and without one the
position-derived encoding degrades to all-`INVALID_VALUE`, so a topology-aware encoder quietly loses
the thing that makes it topology-aware. Setting `standard_1020` fixes it.

In [ ]:
# VERIFY: adapt to however the prepared cache exposes montage info in 0.3.1.
# Intent: assert positions are real before trusting any position-aware encoder.

DATA_SFREQ = 256      # Alljoined-1.6M, matches the hidden Emotiv cohort
N_CHANS    = 32

ENCODER_NATIVE_SFREQ = {'eegnet': None, 'reve': 200, 'luna': 256}

def log_resample_path(model_key):
    native = ENCODER_NATIVE_SFREQ.get(model_key)
    if native is None:
        print(f'{model_key}: no fixed native rate; runs at data rate {DATA_SFREQ} Hz')
    elif native == DATA_SFREQ:
        print(f'{model_key}: native {native} Hz == data {DATA_SFREQ} Hz — no resample')
    else:
        print(f'{model_key}: data {DATA_SFREQ} Hz -> encoder {native} Hz — RESAMPLE, verify it happens')

for k in ENCODER_NATIVE_SFREQ:
    log_resample_path(k)

---
## [5] Shape check, then debug run

`check_model` is a seconds-long shape probe. Run it before spending anything on training.

In [ ]:
from neuralbench import check_model

# VERIFY: signature per docs is check_model(model, modality, task).
# Using the stock baseline first — no gated weights, fastest path to a green pipeline.
# print(check_model(my_model, 'eeg', 'image'))

### Apply the subset, then guard the gallery

**Inert.** These guards need the training-split group ids and the gallery size out of the prepared
cache, and that access pattern is still unknown. They are left in place as the shape of the check
rather than as working code.

Consequence: Stage 1 runs with no gallery guard, so any Top-5 it produces must be read alongside the
gallery size the split actually used.

In [ ]:
# VERIFY: pull the training-split group ids and the test gallery size from the prepared cache.
# train_groups = ...   # subject id per training window
# GALLERY_FULL = ...   # candidate count with no subsetting

# mask = subset_groups(train_groups, spec)
# print(describe(train_groups, mask, spec, gallery_size=GALLERY_FULL))
# assert_gallery_intact(gallery_size=GALLERY_FULL, expected=GALLERY_FULL)
# assert_disjoint(train_image_ids, test_image_ids)   # cross-stimulus split intact

In [ ]:
if RUN_HEAVY:
    !neuralbench eeg image --dataset {DATASET} -m eegnet --debug
else:
    print('skipped: debug run needs the prepared cache')

---
## [6] Chance control

`chance` is a registered model, so this runs through the same scoring path as any other entry rather
than needing a hand-wired predictor. That makes it a genuine check on the retrieval and metric code.

Published chance on THINGS-EEG2 is Top-5 **2.22 ± 0.31**, consistent with its 200-image test gallery
(5/200 = 2.5%). Chance scales as `k / gallery_size`, so the Alljoined figure will differ. What
matters is whether the returned value matches `5 / gallery_size` for whatever gallery the split
produces. If it does not, the scoring path is wrong and nothing downstream is trustworthy.

In [ ]:
if RUN_HEAVY:
    !neuralbench eeg image --dataset {DATASET} -m chance
else:
    print('skipped: chance control needs the prepared cache')

---
## [7] Baseline — EEGNet

0.04 M params, ~2.5 h per seed on THINGS-EEG2 (expect less on the smaller Alljoined corpus).
This becomes **our** local reference on this corpus, since no published Alljoined number exists.

In [ ]:
if RUN_HEAVY and RUN_FULL_TRAIN:
    t0 = time.time()
    !neuralbench eeg image --dataset {DATASET} -m eegnet
    print(f'eegnet took {(time.time() - t0) / 60:.1f} min')
else:
    print('skipped: full training. Set RUN_FULL_TRAIN = True once the debug run is green.')

In [ ]:
if RUN_HEAVY and RUN_FULL_TRAIN:
    !neuralbench eeg image --dataset {DATASET} -m eegnet --plot-cached
else:
    print('skipped: nothing cached to plot yet')

---
## [8] The encoder: REVE

Selected on pretraining breadth, which is the property that should drive this choice.

| Encoder | Corpora | Scale | Channel handling |
|---|---|---|---|
| **REVE** | **92 datasets** | 60,000+ h, 25,000 subjects | 4D positional encoding over 3-D coords |
| LaBraM | 16 datasets | ~2,534 h | fixed 128-ch 10-20 order, unmatched channels dropped |
| BIOT | 6 datasets | - | Conv1d projection to 18 TCP bipolar channels |
| LUNA | TUEG + Siena | TUEG ~26,000 h | learned cross-attention unification |
| CBraMod | TUEG only | TUEG ~26,000 h | accepts any channel count |
| BENDR | TUEG only | TUEG ~26,000 h | fixed 20 channels |

REVE has 92 datasets against LaBraM's 16 and everyone else's 1-6. Three of the six are single-corpus
TUEG models, and TUEG is clinical pathology-screening EEG - far from healthy-subject viewing of
natural images, on top of being narrow.

**Montage handling is not a separate concern to defer.** You cannot pretrain across 92 heterogeneous
datasets without first solving montage invariance; they share no electrode layout. The narrow models
are TUEG-only precisely because one corpus with one montage is the easy case. REVE's positional
encoding is not a convenience bolted on - it is why the wide pretraining was possible.

The alternative has a concrete cost on this track: Alljoined is 32-channel consumer Emotiv. A
fixed-montage encoder either drops unmatched electrodes or interpolates onto a montage that was never
recorded. On 32 channels neither is affordable.

On leakage: REVE's image-task overlap is THINGS-EEG2, which taints its *published* 84.75. It never saw
Alljoined - what we train on, and where the hidden cohort comes from. The model is fine; the number is not.

**Second arm: LUNA.** Native 256 Hz matches Alljoined exactly where REVE resamples from 200 Hz, and it
reaches topology-invariance by a different mechanism. Useful as a check that results are not an artifact
of one encoder's inductive bias.

**Outside the zoo: ST-EEGFormer**, the KU Leuven model that won Challenge 1 in 2025 - roughly 13,300 h
across 11 datasets, open weights, proven in competition. Fewer corpora than REVE and it means leaving the
harness, so: back pocket, not starting point.

**Where the tuning effort probably belongs: the head and loss.** The target space is prescribed (frozen
DINOv2-giant, depth 0.6667, mean-pooled) and the stock config aligns to it with `ClipLoss`
(`norm_kind: y`, `temperature: false`, `symmetric: false`). Published EEG-to-image work (NICE, ATM,
NeuroCLIP) mostly varies the alignment head and objective rather than the backbone. `SigLipLoss` and
`DiffusionPrior` are already in the zoo.

Adaptation order: `-w linear_probe_mean` as a floor, then `-w lora_r4_flatten` and
`-w lora_r32_flatten`. The 2025 winner found full fine-tuning overfit while LoRA did not, and
EEG-FM-Compass found linear probing alone frequently insufficient, so run both ends. Two LoRA ranks
ship, which makes the rank sweep one extra flag rather than a code change. `-w` applies to
foundation models only.

In [ ]:
# login() blocks waiting for input, which stalls Run All. Run it on its own
# once REVE access is granted.
# from huggingface_hub import login; login()

In [ ]:
# REVE needs the gated brain-bzh/reve-base weights. Off Run All until access lands.
# !neuralbench eeg image --dataset {DATASET} -m reve -w linear_probe_mean
# !neuralbench eeg image --dataset {DATASET} -m reve -w lora_r4_flatten
# !neuralbench eeg image --dataset {DATASET} -m reve -w lora_r32_flatten
# !neuralbench eeg image --dataset {DATASET} -m luna -w lora_r4_flatten

---
## [9] Package the submission

Server-side evaluation is **inference-only** — the model arrives trained. A submission is a folder
holding `submission.py` plus weights, zipped.

For Track 1, `predict(X)` takes `(B, C, T)` and returns embeddings **`(B, 1536)`**.
Confirm against the Participation tab before the first real upload.

In [ ]:
SUBMISSION_DIR = ROOT / 'submissions' / 'track01_pipeline_proof'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# torch.save(model.state_dict(), SUBMISSION_DIR / 'weights.pt')

In [ ]:
solver = '''import torch

import benchmark_utils  # noqa: F401 - locates compet_core
from compet_core.base_solver import CompetSolver


class Solver(CompetSolver):
    name = "PipelineProof-T1"

    def load_model(self, meta):
        model = build_model(n_chans=meta["n_chans"], n_times=meta["n_times"],
                            n_outputs=meta["n_outputs"])  # 1536 for track 1
        state = torch.load(meta["weights_dir"] / "weights.pt",
                           map_location=meta["device"])
        model.load_state_dict(state)
        return model.to(meta["device"]).eval()
'''
(SUBMISSION_DIR / 'submission.py').write_text(solver)
print((SUBMISSION_DIR / 'submission.py').read_text())

### Local harness check

Once the starting kit is in hand, `Simulated` needs **zero download** — it runs anywhere,
including the MacBook. Cheapest possible check that our packaging is correct.

In [ ]:
# benchopt install tracks/image_decoding
# benchopt run tracks/image_decoding -d Simulated
# python codabench/ingestion_program/ingestion.py \
#     --submission-dir {SUBMISSION_DIR} --benchmark-dir tracks/image_decoding --datasets Simulated
# python codabench/scoring_program/scoring.py --prediction-dir output/ --output-dir scores/

---
## [10] Inference budget

Hard cap: a full test pass in **under 60 minutes on one H100/H200**. `duration` is a public
leaderboard column. The 2025 Challenge 1 winner lost Challenge 2 entirely to an inference timeout —
this is a real failure mode, not a formality.

In [ ]:
# VERIFY: time a full test pass through predict() on the largest available split.
BUDGET_MIN = 60
# elapsed = ...
# print(f'{elapsed:.1f} min / {BUDGET_MIN} min budget  ({elapsed/BUDGET_MIN:.0%})')

---
## [11] Run record

Rule 4 requires declaring every external corpus **and a compute estimate** in the final method
description. Adopting REVE means inheriting its 92-dataset pretraining corpus. Log it as you go —
reconstructing this in November is miserable.

In [ ]:
import json, datetime

record = {
    'track': '01-eeg-to-image',
    'date': datetime.date.today().isoformat(),
    'dataset': DATASET,
    'model': 'eegnet',
    'metric_key': 'test/full_retrieval/top5_acc_subject-agg',
    'score': None,
    'inference_min': None,
    'external_pretraining': [],   # e.g. REVE -> 92 public EEG datasets
    'compute_estimate_gpu_h': None,
    'notes': 'pipeline proof; no published Alljoined reference to match',
}
path = SAVE_DIR / f"run_{record['date']}_t01.json"
path.write_text(json.dumps(record, indent=2))
print(path)